# Comparação: Cenário B (Fase 1) vs B+ (Fase 2)

**B_strict** = replica o Cenário B da Fase 1 a partir dos dados brutos  
→ contínuas + ordinais + PAI_AUSENTE (sem flags derivadas, sem FAIXAETAMAE)

**B_plus** = pipeline do Caê (B_strict + PNTARDIO + HISTPERDAFETAL + PRIMIPARA + FAIXAETAMAE OHE)

Ambos usam os mesmos hiperparâmetros (Exp1_conservador do AG 03) e threshold 0.40.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, fbeta_score, precision_score, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

from src.cleaning import clean_dataset
from src.transformers import FeatureEngineer
from src.pipeline_factory import build_pipeline
from src.constants import (
    DEFAULT_THRESHOLD, RANDOM_STATE, HISTGB_PARAMS,
    NUMERIC_COLUMNS, ESCMAE2010_ORDINAL_COLUMN, KOTELCHUCK_ORDINAL_COLUMN,
)

## 1. Dados

In [ ]:
df_raw = pd.read_parquet('../data/df_model_raw.parquet')
df_clean = clean_dataset(df_raw)

y = df_clean['PREMATURO']
X = df_clean.drop(columns=['PREMATURO'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Prevalência prematuro (treino): {y_train.mean():.3%}')
print(f'Colunas disponíveis: {X_train.columns.tolist()}')

## 2. B_strict — Cenário B da Fase 1 (sem flags, sem FAIXAETAMAE)

In [ ]:
def build_pipeline_b_strict(random_state: int = RANDOM_STATE) -> Pipeline:
    """Replica o Cenário B da Fase 1: ordinais + contínuas + PAI_AUSENTE."""
    numeric_cols = list(NUMERIC_COLUMNS) + [ESCMAE2010_ORDINAL_COLUMN, KOTELCHUCK_ORDINAL_COLUMN]
    passthrough_cols = ['PAI_AUSENTE']  # sem PNTARDIO, HISTPERDAFETAL, PRIMIPARA

    preprocessor = ColumnTransformer(
        transformers=[
            ('numeric', StandardScaler(), numeric_cols),
            ('passthrough', 'passthrough', passthrough_cols),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )
    return Pipeline([
        ('feature_engineer', FeatureEngineer()),
        ('preprocessor', preprocessor),
        ('histgradientboostingclassifier', HistGradientBoostingClassifier(
            random_state=random_state, **HISTGB_PARAMS
        )),
    ])

## 3. Treinar ambos

In [ ]:
weights = compute_sample_weight('balanced', y_train)
fit_kwargs = {'histgradientboostingclassifier__sample_weight': weights}

print('Treinando B_strict...')
pipe_strict = build_pipeline_b_strict()
pipe_strict.fit(X_train, y_train, **fit_kwargs)

print('Treinando B_plus...')
pipe_plus = build_pipeline()
pipe_plus.fit(X_train, y_train, **fit_kwargs)

print('Pronto.')

## 4. Avaliar @ threshold 0.40

In [ ]:
def evaluate(pipe, X_test, y_test, threshold=DEFAULT_THRESHOLD, name=''):
    prob = pipe.predict_proba(X_test)[:, 1]
    pred = (prob >= threshold).astype(int)
    tp = ((pred == 1) & (y_test == 1)).sum()
    fn = ((pred == 0) & (y_test == 1)).sum()
    fp = ((pred == 1) & (y_test == 0)).sum()
    return {
        'modelo': name,
        'recall':    round(recall_score(y_test, pred), 4),
        'f2':        round(fbeta_score(y_test, pred, beta=2), 4),
        'precisão':  round(precision_score(y_test, pred, zero_division=0), 4),
        'roc_auc':   round(roc_auc_score(y_test, prob), 4),
        'TP': int(tp), 'FN': int(fn), 'FP': int(fp),
        'features': pipe['preprocessor'].get_feature_names_out().shape[0],
    }

resultados = pd.DataFrame([
    evaluate(pipe_strict, X_test, y_test, name='B_strict (Fase 1)'),
    evaluate(pipe_plus,   X_test, y_test, name='B_plus   (Fase 2)'),
])

print(resultados.to_string(index=False))

## 5. Referência da Fase 1 (baseline calibrado)

In [ ]:
with open('../results/metrics/best_model_operational_metrics.json') as f:
    baseline = json.load(f)

m = baseline['metrics']
print('=== Referência Fase 1 (best_model_calibrated.pkl) ===')
print(f"  recall:   {m['recall']:.4f}")
print(f"  f2:       {m['f2']:.4f}")
print(f"  precisão: {m['precision']:.4f}")
print(f"  roc_auc:  {m['roc_auc']:.4f}")
print(f"  TP: {m['tp']}  FN: {m['fn']}  FP: {m['fp']}")